# Fluid-rock reaction relay demo

Notebook for the decompression relay function. Last updated 10 Nov 2026.
Start with dependencies:

In [1]:
import numpy as np
import pyDEW
import os
import matplotlib.pyplot as plt
from thermoengine import model
from copy import deepcopy
import pandas as pd
import linecache
from scipy.interpolate import LinearNDInterpolator
from timeit import default_timer as timer
from datetime import datetime, timezone
import warnings

# Additional functions for pyDEW\
import pyDEW_functions as pDF

# Structural functions for the relay
import relay_functions as rF

# Folder to export results
if not os.path.exists("./data"):
    os.makedirs("./data")

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None


Create our pyDEW system (required for DATA0 and DATA1 files).

In [2]:
# Adjust here the basis species in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2 directly to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)
fO2Database = model.Database()


Create the reactants that will fill our column.

In [3]:
# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
           'moles': 0.2,
           'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

spnDict = {'reactant': 'SPINEL',
           'moles': 0.04}

startingReactant = [olvDict, opxDict, cpxDict, spnDict]


Create test case inputs. Commented-out lines are interval resolution tests.

In [4]:
# TEST CASE 1: ISOTHERMAL, ISOBARIC
case1_100 = rF.createUniformInput(startingReactant, dew_system, 1000, [20000]*101, [500+273.15]*101)
# case1_200 = rF.createUniformInput(startingReactant, dew_system, 500, [20000]*201, [500+273.15]*201)
# case1_050 = rF.createUniformInput(startingReactant, dew_system, 2000, [20000]*51, [500+273.15]*51)
# case1_400 = rF.createUniformInput(startingReactant, dew_system, 250, [20000]*401, [500+273.15]*401)

# TEST CASE 2: ISOTHERMAL ONLY
case2_100 = rF.createUniformInput(startingReactant, dew_system, 1000, np.linspace(25000, 15000, 101), [500+273.15]*101)
# case2_200 = rF.createUniformInput(startingReactant, dew_system, 500, np.linspace(25000, 15000, 201), [500+273.15]*201)
# case2_050 = rF.createUniformInput(startingReactant, dew_system, 2000, np.linspace(25000, 15000, 51), [500+273.15]*51)
# case2_400 = rF.createUniformInput(startingReactant, dew_system, 250, np.linspace(25000, 15000, 401), [500+273.15]*401)

# TEST CASE 3: ISOBARIC ONLY
case3_100 = rF.createUniformInput(startingReactant, dew_system, 1000, [20000]*101, np.linspace(700+273.15, 300+273.15, 101))
# case3_200 = rF.createUniformInput(startingReactant, dew_system, 500, [20000]*201, np.linspace(700+273.15, 300+273.15, 201))
# case3_050 = rF.createUniformInput(startingReactant, dew_system, 2000, [20000]*51, np.linspace(700+273.15, 300+273.15, 51))
# case3_400 = rF.createUniformInput(startingReactant, dew_system, 250, [20000]*401, np.linspace(700+273.15, 300+273.15, 401))

# TEST CASE 4
case4_100 = rF.createUniformInput(startingReactant, dew_system, 1000, np.linspace(25000, 15000, 101), np.linspace(700+273.15, 300+273.15, 101))
# case4_200 = rF.createUniformInput(startingReactant, dew_system, 500, np.linspace(25000, 15000, 201), np.linspace(700+273.15, 300+273.15, 201))
# case4_050 = rF.createUniformInput(startingReactant, dew_system, 2000, np.linspace(25000, 15000, 51), np.linspace(700+273.15, 300+273.15, 51))
# case4_400 = rF.createUniformInput(startingReactant, dew_system, 250, np.linspace(25000, 15000, 401), np.linspace(700+273.15, 300+273.15, 401))


tests = {
    '1_100': case1_100,
    '2_100': case2_100,
    '3_100': case3_100,
    '4_100': case4_100
        }


Run relay for each of the three fluids.

In [5]:
fluids = ['DMM', 'MOR', 'SED']
for i in range(len(tests)):
    for j in range(len(fluids)):
        # Timer to keep track of calculation times
        start = timer()
        
        # Pick our fluid
        fluidType = fluids[j]
        
        # Pick our starting setup
        testLabel = list(tests.keys())[i]
        startingComposition = tests[testLabel]

        print(datetime.now())
        print(f'Conditions: fluid: {fluidType}, column: {testLabel}')
        
        # Starting PT is collected from the starting composition input
        startingP = startingComposition['Pressure (bar)'].iloc[0]
        startingT = startingComposition['Temperature (K)'].iloc[0]
        
        # Create our fluid
        if fluidType == 'DMM':
            fluidComposition = rF.createHybridFluid([1,0,0], startingP, startingT)
        elif fluidType == 'MOR':
            fluidComposition = rF.createHybridFluid([0,1,0], startingP, startingT)
        elif fluidType == 'SED':
            fluidComposition = rF.createHybridFluid([0,0,1], startingP, startingT)

        # Fluid-rock ratio at which to assume no further reaction
        terminalFRR = 0.1

        # Run relay
        rF.decompressionRelay(startingComposition, fluidComposition[0], fluidComposition[1],
                              dew_system, fO2Database, timesteps=10, solventMass=1000,
                              terminalFRR=terminalFRR,
                              exportPath=f'{testLabel}_{fluidType}_{terminalFRR}',
                              TCheck=True)
        print(datetime.now())



2026-11-10 16:55:32.425873
Conditions: fluid: DMM, column: 1_100
-----RELAY STARTING-----
Relay function initiated
1000
Starting from scratch...


/opt/conda/lib/python3.10/site-packages/pyDEW/eq6_output.py:82: UserWarning: The species concentration output could not be read.
  warn("The species concentration output could not be read.")


AttributeError: 'NoneType' object has no attribute 'index'